In [ ]:
import mne
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from moabb.datasets import Zuo2025
import plotly.io as pio

pio.renderers.default = "vscode"

In [ ]:


dataset = Zuo2025()
data = dataset.get_data()
participantes_todos = list(data.keys())

sessoes_por_sujeito = {p: len(data[p]) for p in participantes_todos}

n_sessoes_esperado = max(sessoes_por_sujeito.values())

sujeitos_completos = sorted(
    [p for p, n in sessoes_por_sujeito.items() if n == n_sessoes_esperado]
)
sujeitos_incompletos = sorted(
    [p for p, n in sessoes_por_sujeito.items() if n < n_sessoes_esperado]
)

total_sessoes_esperadas = n_sessoes_esperado * len(participantes_todos)
total_sessoes_presentes = sum(sessoes_por_sujeito.values())
total_sessoes_faltando = total_sessoes_esperadas - total_sessoes_presentes

print("=" * 70)
print("RELATÓRIO DE INTEGRIDADE DO DATASET Zuo2025")
print("=" * 70)
print(f"Sessões esperadas por sujeito (dataset completo): {n_sessoes_esperado}")
print(f"Total de sujeitos:                                {len(participantes_todos)}")
print(f"Sessões esperadas no total:                        {total_sessoes_esperadas}")
print(f"Sessões efetivamente presentes:                    {total_sessoes_presentes}")
print(f"Sessões faltando no total:                         {total_sessoes_faltando}")
print("-" * 70)
print(f"Sujeitos com dataset COMPLETO   ({len(sujeitos_completos)}): {sujeitos_completos}")
print(f"Sujeitos com dataset INCOMPLETO ({len(sujeitos_incompletos)}):")
for p in sujeitos_incompletos:
    faltando = n_sessoes_esperado - sessoes_por_sujeito[p]
    sessoes_presentes = list(data[p].keys())
    print(f"   - Sujeito {p}: {sessoes_por_sujeito[p]}/{n_sessoes_esperado} sessões "
          f"({faltando} faltando) | sessões presentes: {sessoes_presentes}")
print("=" * 70)
print("\nApenas os sujeitos com dataset COMPLETO estarão disponíveis "
      "para seleção no gráfico interativo abaixo.\n")


cache_epochs = {}  

tmin, tmax = dataset.interval


def carregar_epochs(participante, sessao):
    chave = (participante, sessao)
    if chave in cache_epochs:
        return cache_epochs[chave]

    run_key = list(data[participante][sessao].keys())[0]
    raw = data[participante][sessao][run_key]

    events, event_id = mne.events_from_annotations(raw)
    epochs = mne.Epochs(raw, events, event_id=event_id,
                         tmin=tmin, tmax=tmax, baseline=None, preload=True)
    epochs.pick('eeg')

    cache_epochs[chave] = epochs
    return epochs


dd_sujeito = widgets.Dropdown(options=sujeitos_completos, description='Sujeito:')
dd_sessao = widgets.Dropdown(description='Sessão:')
dd_classe = widgets.Dropdown(description='Classe:')
dd_trial = widgets.Dropdown(description='Trial:')

status_label = widgets.HTML(
    value=(f"<b>Sujeitos completos disponíveis:</b> {len(sujeitos_completos)} "
           f"&nbsp;|&nbsp; <b>Sujeitos incompletos (excluídos):</b> "
           f"{len(sujeitos_incompletos)} {sujeitos_incompletos}")
)

fig = go.FigureWidget()
fig.update_layout(
    xaxis_title="Tempo (s)",
    yaxis_title="Amplitude (µV)",
    legend_title="Canal",
    margin=dict(l=10, r=10, t=60, b=10),
    width=1300,
    height=500
)

ch_names_atual = []

def plotar():
    participante = dd_sujeito.value
    sessao = dd_sessao.value
    classe = dd_classe.value
    trial_num = dd_trial.value
    if None in (participante, sessao, classe, trial_num):
        return

    epochs = carregar_epochs(participante, sessao)
    ep_classe = epochs[classe]
    trial_data = ep_classe[trial_num - 1].get_data()[0]
    times = ep_classe.times

    global ch_names_atual
    with fig.batch_update():
        if ch_names_atual != epochs.ch_names:
            fig.data = []
            for ch in epochs.ch_names:
                fig.add_trace(go.Scatter(x=times, y=[], mode='lines', name=ch))
            ch_names_atual = epochs.ch_names[:]
        for i in range(len(ch_names_atual)):
            fig.data[i].x = times
            fig.data[i].y = trial_data[i]
        fig.layout.title = (f"Sujeito {participante} | Sessão {sessao} | "
                             f"Classe: {classe} | Trial {trial_num}")


def atualizar_sessoes(change=None):
    sessoes = list(data[dd_sujeito.value].keys())
    dd_sessao.options = sessoes
    if sessoes:
        dd_sessao.value = sessoes[0]
    atualizar_classes()


def atualizar_classes(change=None):
    participante, sessao = dd_sujeito.value, dd_sessao.value
    if participante is None or sessao is None:
        return
    epochs = carregar_epochs(participante, sessao)
    classes = list(epochs.event_id.keys())
    dd_classe.options = classes
    if classes:
        dd_classe.value = classes[0]
    atualizar_trials()


def atualizar_trials(change=None):
    participante, sessao, classe = dd_sujeito.value, dd_sessao.value, dd_classe.value
    if None in (participante, sessao, classe):
        return
    epochs = carregar_epochs(participante, sessao)
    n_trials = len(epochs[classe])
    trials = list(range(1, n_trials + 1))
    dd_trial.options = trials
    if trials:
        dd_trial.value = trials[0]
    plotar()


dd_sujeito.observe(atualizar_sessoes, names='value')
dd_sessao.observe(atualizar_classes, names='value')
dd_classe.observe(atualizar_trials, names='value')
dd_trial.observe(lambda change: plotar(), names='value')

if sujeitos_completos:
    atualizar_sessoes()
else:
    print("Nenhum sujeito com dataset completo foi encontrado — "
          "verifique a conexão com o Figshare e tente novamente.")

controles = widgets.HBox([dd_sujeito, dd_sessao, dd_classe, dd_trial])
display(status_label, controles, fig)